In [8]:
%%writefile md5_gpu.cu
#include <stdio.h>
#include <stdint.h>
#include <string.h>

// Constantes MD5
__device__ __constant__ uint32_t K[64] = {
    0xd76aa478, 0xe8c7b756, 0x242070db, 0xc1bdceee,
    0xf57c0faf, 0x4787c62a, 0xa8304613, 0xfd469501,
    0x698098d8, 0x8b44f7af, 0xffff5bb1, 0x895cd7be,
    0x6b901122, 0xfd987193, 0xa679438e, 0x49b40821,
    0xf61e2562, 0xc040b340, 0x265e5a51, 0xe9b6c7aa,
    0xd62f105d, 0x02441453, 0xd8a1e681, 0xe7d3fbc8,
    0x21e1cde6, 0xc33707d6, 0xf4d50d87, 0x455a14ed,
    0xa9e3e905, 0xfcefa3f8, 0x676f02d9, 0x8d2a4c8a,
    0xfffa3942, 0x8771f681, 0x6d9d6122, 0xfde5380c,
    0xa4beea44, 0x4bdecfa9, 0xf6bb4b60, 0xbebfbc70,
    0x289b7ec6, 0xeaa127fa, 0xd4ef3085, 0x04881d05,
    0xd9d4d039, 0xe6db99e5, 0x1fa27cf8, 0xc4ac5665,
    0xf4292244, 0x432aff97, 0xab9423a7, 0xfc93a039,
    0x655b59c3, 0x8f0ccc92, 0xffeff47d, 0x85845dd1,
    0x6fa87e4f, 0xfe2ce6e0, 0xa3014314, 0x4e0811a1,
    0xf7537e82, 0xbd3af235, 0x2ad7d2bb, 0xeb86d391
};

__device__ __constant__ uint32_t S[64] = {
    7,12,17,22, 7,12,17,22, 7,12,17,22, 7,12,17,22,
    5, 9,14,20, 5, 9,14,20, 5, 9,14,20, 5, 9,14,20,
    4,11,16,23, 4,11,16,23, 4,11,16,23, 4,11,16,23,
    6,10,15,21, 6,10,15,21, 6,10,15,21, 6,10,15,21
};

__device__ void md5(const uint8_t *msg, uint32_t len, uint8_t *digest) {
    uint32_t a0 = 0x67452301, b0 = 0xefcdab89;
    uint32_t c0 = 0x98badcfe, d0 = 0x10325476;

    // Padding
    uint8_t buf[64] = {0};
    for (uint32_t i = 0; i < len; i++) buf[i] = msg[i];
    buf[len] = 0x80;
    uint64_t bitlen = (uint64_t)len * 8;
    for (int i = 0; i < 8; i++)
        buf[56 + i] = (bitlen >> (8 * i)) & 0xff;

    // Procesar bloque
    uint32_t *M = (uint32_t *)buf;
    uint32_t A = a0, B = b0, C = c0, D = d0;

    for (int i = 0; i < 64; i++) {
        uint32_t F, g;
        if (i < 16)      { F = (B & C) | (~B & D); g = i; }
        else if (i < 32) { F = (D & B) | (~D & C); g = (5*i+1) % 16; }
        else if (i < 48) { F = B ^ C ^ D;           g = (3*i+5) % 16; }
        else             { F = C ^ (B | ~D);         g = (7*i) % 16; }

        F = F + A + K[i] + M[g];
        A = D; D = C; C = B;
        B = B + ((F << S[i]) | (F >> (32 - S[i])));
    }

    a0 += A; b0 += B; c0 += C; d0 += D;

    // Output
    uint32_t *out = (uint32_t *)digest;
    out[0] = a0; out[1] = b0; out[2] = c0; out[3] = d0;
}

__global__ void md5_kernel(const char *input, int len, uint8_t *output) {
    md5((const uint8_t *)input, len, output);
}

int main(int argc, char *argv[]) {
    if (argc < 2) {
        printf("Uso: ./md5_gpu <string>\n");
        return 1;
    }

    const char *input = argv[1];
    int len = strlen(input);

    char *d_input;
    uint8_t *d_output;
    uint8_t h_output[16];

    cudaMalloc(&d_input, len);
    cudaMalloc(&d_output, 16);
    cudaMemcpy(d_input, input, len, cudaMemcpyHostToDevice);

    md5_kernel<<<1, 1>>>(d_input, len, d_output);
    cudaDeviceSynchronize();

    cudaMemcpy(h_output, d_output, 16, cudaMemcpyDeviceToHost);

    printf("Input:  %s\n", input);
    printf("MD5:    ");
    for (int i = 0; i < 16; i++) printf("%02x", h_output[i]);
    printf("\n");

    cudaFree(d_input);
    cudaFree(d_output);
    return 0;
}

Writing md5_gpu.cu


In [9]:
!nvcc md5_gpu.cu -o md5_gpu && ./md5_gpu "hola mundo"

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Input:  hola mundo
MD5:    0ad066a5d29f3f2a2a1c7c17dd082a79
